# 00 — Kiểm kê nguồn và cấu trúc dữ liệu thô

## Mục tiêu và phạm vi

Notebook này kiểm kê các tệp dữ liệu thô trong `data/raw/` trước khi làm sạch. Các bước gồm lập danh mục nguồn, tính mã kiểm tra SHA-256, khảo sát cấu trúc cột và kiểm tra khóa của từng bộ dữ liệu.

Ngày truy cập `2026-09-21` trong danh mục là mốc được ghi trong tài liệu bàn giao, không phải ngày tự động lấy khi chạy notebook. Vì lịch sử tải chưa lưu đầy đủ commit hoặc đường dẫn tải chính xác, thông tin nguồn không đủ để khẳng định các tệp đang lưu trùng với phiên bản trực tuyến hiện tại. SHA-256 giúp nhận diện và đối chiếu nội dung tệp đã được dùng; nó không xác minh tính đúng đắn của dữ liệu.

Notebook đọc trực tiếp dữ liệu thô và ghi các báo cáo kiểm kê. Riêng `SOURCE_MANIFEST.csv` và `checksums_sha256.txt` được cập nhật trong `data/raw/`; nội dung các tệp dữ liệu nguồn không bị chỉnh sửa.

In [1]:
from pathlib import Path
import sys, tarfile
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import RAW, QUALITY, ensure_output_dirs, sha256_file, iter_coinmetrics, write_csv_atomic

ensure_output_dirs()
pd.set_option("display.max_colwidth", 100)

## Danh mục nguồn và mục đích sử dụng

Mỗi dòng trong danh mục tương ứng với một tệp nguồn, gồm đường dẫn trong dự án, tên nguồn, liên kết tham khảo, giấy phép được ghi nhận, đơn vị của một dòng dữ liệu, khóa dự kiến và mục đích sử dụng.

Chương trình kiểm tra sự tồn tại của từng tệp, ghi kích thước và tính SHA-256. Nếu thiếu tệp, bước xử lý dừng lại. Danh mục được lưu tại `data/raw/SOURCE_MANIFEST.csv`, kèm tệp `checksums_sha256.txt` để đối chiếu nội dung về sau.

In [2]:
ACCESS_DATE = "2026-09-21"
sources = [
    {
        "raw_path": "crypto/coinmetrics_raw.tar.gz",
        "source_name": "Coin Metrics Community Data",
        "source_url": "https://github.com/coinmetrics/data",
        "evidence_url": "https://github.com/coinmetrics/data/tree/master/csv",
        "license": "CC BY-NC 4.0 theo README nguồn",
        "grain": "một coin - một ngày trong từng CSV thành viên",
        "candidate_key": "ticker,time",
        "use": "giá, vốn hóa current/estimated, reported spot volume và chỉ số on-chain cryptocurrency",
        "provenance_limit": "Không lưu commit/URL tải chính xác; dùng SHA-256 snapshot local.",
    },
    {
        "raw_path": "macro/vix_daily.csv", "source_name": "CBOE VIX - Core Datasets",
        "source_url": "https://github.com/datasets/finance-vix",
        "evidence_url": "https://github.com/datasets/finance-vix/blob/main/data/vix-daily.csv",
        "license": "PDDL", "grain": "một ngày giao dịch", "candidate_key": "DATE",
        "use": "mức VIX và chế độ rủi ro", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "macro/brent_daily.csv", "source_name": "Brent Oil - Core Datasets",
        "source_url": "https://github.com/datasets/oil-prices",
        "evidence_url": "https://github.com/datasets/oil-prices/blob/main/data/brent-daily.csv",
        "license": "PDDL", "grain": "một ngày giao dịch", "candidate_key": "Date",
        "use": "giá dầu Brent USD/thùng", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "macro/fx_daily.csv", "source_name": "Federal Reserve FX - Core Datasets",
        "source_url": "https://github.com/datasets/exchange-rates",
        "evidence_url": "https://github.com/datasets/exchange-rates/blob/main/exchange_rates_flow.py",
        "license": "PDDL", "grain": "một đồng tiền - một ngày giao dịch",
        "candidate_key": "Country,Date", "use": "cấu phần dựng lại DXY",
        "provenance_limit": "Package upstream đã đảo AUD/EUR/NZD/GBP về ngoại tệ trên 1 USD; cần kiểm chứng DXY độc lập trước kết luận chính.",
    },
    {
        "raw_path": "macro/sp500_monthly.csv", "source_name": "S&P 500 - Core Datasets",
        "source_url": "https://github.com/datasets/s-and-p-500",
        "evidence_url": "https://github.com/datasets/s-and-p-500/tree/main/data",
        "license": "PDDL", "grain": "một tháng", "candidate_key": "Date",
        "use": "mức S&P 500 và PE10", "provenance_limit": "Không dùng cột lãi suất placeholder làm nguồn chính.",
    },
    {
        "raw_path": "macro/us10y_monthly.csv", "source_name": "US 10Y Treasury - Core Datasets",
        "source_url": "https://github.com/datasets/bond-yields-us-10y",
        "evidence_url": "https://github.com/datasets/bond-yields-us-10y/tree/main/data",
        "license": "PDDL", "grain": "một tháng", "candidate_key": "Date",
        "use": "lợi suất trái phiếu Mỹ 10 năm", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "macro/gold_monthly.csv", "source_name": "Gold Prices - Core Datasets",
        "source_url": "https://github.com/datasets/gold-prices",
        "evidence_url": "https://github.com/datasets/gold-prices/tree/main/data",
        "license": "PDDL", "grain": "một tháng", "candidate_key": "Date",
        "use": "giá vàng USD/ounce", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "macro/cpi_us_monthly.csv", "source_name": "US CPI - Core Datasets",
        "source_url": "https://github.com/datasets/cpi-us",
        "evidence_url": "https://github.com/datasets/cpi-us/tree/main/data",
        "license": "PDDL", "grain": "một tháng", "candidate_key": "Date",
        "use": "chỉ số CPI Mỹ; tự tính MoM và YoY từ Index", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "country/wb_cpi.csv", "source_name": "World Bank WDI inflation",
        "source_url": "https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG",
        "evidence_url": "https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG",
        "license": "CC BY 4.0", "grain": "một ISO3 - một năm", "candidate_key": "Country Code,Year",
        "use": "lạm phát CPI hằng năm (%)", "provenance_limit": "URL tải/mirror ban đầu không được lưu; không gọi cột này là mức CPI.",
    },
    {
        "raw_path": "country/wb_gdp.csv", "source_name": "World Bank GDP mirror",
        "source_url": "https://github.com/datasets/gdp",
        "evidence_url": "https://github.com/datasets/gdp/tree/main/data",
        "license": "CC BY 4.0", "grain": "một ISO3 - một năm", "candidate_key": "Country Code,Year",
        "use": "GDP hiện hành USD", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "country/wb_population.csv", "source_name": "World Bank population mirror",
        "source_url": "https://github.com/datasets/population",
        "evidence_url": "https://github.com/datasets/population/tree/main/data",
        "license": "CC BY 4.0", "grain": "một ISO3 - một năm", "candidate_key": "Country Code,Year",
        "use": "dân số", "provenance_limit": "Không lưu commit upstream chính xác.",
    },
    {
        "raw_path": "country/country_codes.csv", "source_name": "Core Datasets country codes",
        "source_url": "https://github.com/datasets/country-codes",
        "evidence_url": "https://github.com/datasets/country-codes/tree/main/data",
        "license": "PDDL", "grain": "một mã ISO3", "candidate_key": "ISO3166-1-Alpha-3",
        "use": "dimension quốc gia và mã châu lục", "provenance_limit": "Chuỗi Continent='NA' là Bắc Mỹ, không phải missing.",
    },
]
manifest = pd.DataFrame(sources)
manifest["snapshot_access_date"] = ACCESS_DATE
manifest["file_exists"] = manifest.raw_path.map(lambda x: (RAW / x).exists())
manifest["size_bytes"] = manifest.raw_path.map(lambda x: (RAW / x).stat().st_size if (RAW / x).exists() else pd.NA)
manifest["sha256"] = manifest.raw_path.map(lambda x: sha256_file(RAW / x) if (RAW / x).exists() else pd.NA)
if not manifest.file_exists.all():
    raise FileNotFoundError(manifest.loc[~manifest.file_exists, "raw_path"].tolist())
write_csv_atomic(manifest, RAW / "SOURCE_MANIFEST.csv")
(RAW / "checksums_sha256.txt").write_text(
    "".join(f"{row.sha256}  {row.raw_path}\n" for row in manifest.itertuples()), encoding="utf-8"
)
manifest[["raw_path", "source_name", "grain", "candidate_key", "sha256"]]

,raw_path,source_name,grain,candidate_key,sha256
0,crypto/coinmetrics_raw.tar.gz,Coin Metrics Community Data,một coin - một ngày trong từng CSV thành viên,"ticker,time",0eac99faeba64eac71790c1f9f551a171e8803269540cbe83939be5e0a9cc23c
1,macro/vix_daily.csv,CBOE VIX - Core Datasets,một ngày giao dịch,DATE,50fd00ef48a9e51067407a92f6839307d601b644d8ac08ba711ee1ef77e71367
2,macro/brent_daily.csv,Brent Oil - Core Datasets,một ngày giao dịch,Date,f74e3fb2bc2af60e469b61362ae85574613f47885ad2486b39a81fe4dabd6c8f
3,macro/fx_daily.csv,Federal Reserve FX - Core Datasets,một đồng tiền - một ngày giao dịch,"Country,Date",480ce31805befecb1bf6315de3ed7de3882f8da84b4240aa4084616399ca1bf2
4,macro/sp500_monthly.csv,S&P 500 - Core Datasets,một tháng,Date,e808bc8fad8ed0c6f658eaca27b136c63b51abfa3f5183cbe388138b589f42df
5,macro/us10y_monthly.csv,US 10Y Treasury - Core Datasets,một tháng,Date,2ce54ee178023699587e2c01a1bbe8878aed45a22b738c8e2ec71d63b6b51b46
6,macro/gold_monthly.csv,Gold Prices - Core Datasets,một tháng,Date,6a48963956282b942bf7dca36fe9e1ae5d4a51e0266ae4e697acbf7a28db0f68
7,macro/cpi_us_monthly.csv,US CPI - Core Datasets,một tháng,Date,3593ec66b5293d9cbbaba0785e87f42c5b2a717282cc96561b01648c4089b6c2
8,country/wb_cpi.csv,World Bank WDI inflation,một ISO3 - một năm,"Country Code,Year",182baa44c536079bea07bb8700c37c85441c3d7d5f8e6dd68fb92fcb4a7815e8
9,country/wb_gdp.csv,World Bank GDP mirror,một ISO3 - một năm,"Country Code,Year",f0a8408195646dbb1a9d7fc4424e2d302ee5380d0ec8834793f12ca25cbd7e2c


## Khảo sát cấu trúc, số dòng, thời gian và giá trị thiếu

Báo cáo `raw_inventory.csv` ghi số dòng và thông tin tổng quát của từng tệp. Với các CSV thông thường, khoảng thời gian được lấy từ cột `Date`, `DATE` hoặc `Year`, nếu có. Với kho nén Coin Metrics, notebook đếm số CSV thành viên và tổng số dòng; khoảng ngày của từng tài sản sẽ được xác định ở notebook 01.

Báo cáo `raw_schema_profile.csv` mô tả từng cột: kiểu dữ liệu khi đọc, số giá trị thiếu, số giá trị khác nhau và một ví dụ không thiếu. Đây là thông tin khảo sát trước khi chuẩn hóa, nên kiểu dữ liệu khi đọc chưa nhất thiết là kiểu dùng trong phân tích.

Khi đọc `country_codes.csv`, chuỗi `NA` được giữ nguyên vì đó là mã châu lục Bắc Mỹ; ô rỗng mới được xem là giá trị thiếu.

In [3]:
date_candidates = ["Date", "DATE", "Year"]
inventory_rows, schema_rows = [], []
for row in manifest.itertuples():
    path = RAW / row.raw_path
    if path.suffixes[-2:] == [".tar", ".gz"]:
        coin_rows = 0
        members = 0
        for ticker, frame in iter_coinmetrics(path):
            members += 1
            coin_rows += len(frame)
            for col in frame.columns:
                schema_rows.append({
                    "raw_path": row.raw_path, "member": ticker, "column": col,
                    "dtype_read": str(frame[col].dtype), "rows": len(frame),
                    "missing": int(frame[col].isna().sum()), "unique_non_null": int(frame[col].nunique(dropna=True)),
                    "example": next((str(v) for v in frame[col].dropna().head(1)), ""),
                })
        inventory_rows.append({
            "raw_path": row.raw_path, "rows": coin_rows, "columns": pd.NA,
            "members": members, "date_min": pd.NA, "date_max": pd.NA,
        })
        continue

    keep_default = False if path.name == "country_codes.csv" else True
    na_values = [""] if path.name == "country_codes.csv" else None
    frame = pd.read_csv(path, low_memory=False, keep_default_na=keep_default, na_values=na_values)
    date_col = next((c for c in date_candidates if c in frame.columns), None)
    if date_col == "Year":
        date_min, date_max = int(frame[date_col].min()), int(frame[date_col].max())
    elif date_col:
        parsed = pd.to_datetime(frame[date_col], errors="coerce")
        date_min, date_max = parsed.min(), parsed.max()
    else:
        date_min = date_max = pd.NA
    inventory_rows.append({
        "raw_path": row.raw_path, "rows": len(frame), "columns": frame.shape[1],
        "members": 1, "date_min": date_min, "date_max": date_max,
    })
    for col in frame.columns:
        schema_rows.append({
            "raw_path": row.raw_path, "member": "", "column": col,
            "dtype_read": str(frame[col].dtype), "rows": len(frame),
            "missing": int(frame[col].isna().sum()), "unique_non_null": int(frame[col].nunique(dropna=True)),
            "example": next((str(v) for v in frame[col].dropna().head(1)), ""),
        })

inventory = pd.DataFrame(inventory_rows).merge(manifest[["raw_path", "sha256"]], on="raw_path", validate="one_to_one")
schema = pd.DataFrame(schema_rows)
write_csv_atomic(inventory, QUALITY / "raw_inventory.csv")
write_csv_atomic(schema, QUALITY / "raw_schema_profile.csv")
print(f"Tổng dòng raw: {inventory.rows.sum():,.0f}; số snapshot: {len(inventory)}")
inventory

Tổng dòng raw: 530,018; số snapshot: 12


,raw_path,rows,columns,members,date_min,date_max,sha256
0,crypto/coinmetrics_raw.tar.gz,189355,<NA>,63,<NA>,<NA>,0eac99faeba64eac71790c1f9f551a171e8803269540cbe83939be5e0a9cc23c
1,macro/vix_daily.csv,9276,5,1,1990-01-02 00:00:00,2026-09-18 00:00:00,50fd00ef48a9e51067407a92f6839307d601b644d8ac08ba711ee1ef77e71367
2,macro/brent_daily.csv,9087,2,1,1987-05-20 00:00:00,2026-09-15 00:00:00,f74e3fb2bc2af60e469b61362ae85574613f47885ad2486b39a81fe4dabd6c8f
3,macro/fx_daily.csv,273261,3,1,1971-01-04 00:00:00,2026-09-11 00:00:00,480ce31805befecb1bf6315de3ed7de3882f8da84b4240aa4084616399ca1bf2
4,macro/sp500_monthly.csv,1868,10,1,1871-01-01 00:00:00,2026-08-01 00:00:00,e808bc8fad8ed0c6f658eaca27b136c63b51abfa3f5183cbe388138b589f42df
5,macro/us10y_monthly.csv,880,2,1,1953-04-01 00:00:00,2026-07-01 00:00:00,2ce54ee178023699587e2c01a1bbe8878aed45a22b738c8e2ec71d63b6b51b46
6,macro/gold_monthly.csv,2324,2,1,1833-01-01 00:00:00,2026-08-01 00:00:00,6a48963956282b942bf7dca36fe9e1ae5d4a51e0266ae4e697acbf7a28db0f68
7,macro/cpi_us_monthly.csv,1362,3,1,1913-01-01 00:00:00,2026-07-01 00:00:00,3593ec66b5293d9cbbaba0785e87f42c5b2a717282cc96561b01648c4089b6c2
8,country/wb_cpi.csv,11182,4,1,1960,2024,182baa44c536079bea07bb8700c37c85441c3d7d5f8e6dd68fb92fcb4a7815e8
9,country/wb_gdp.csv,13979,4,1,1960,2023,f0a8408195646dbb1a9d7fc4424e2d302ee5380d0ec8834793f12ca25cbd7e2c


## Kiểm tra khóa của dữ liệu thô

Khóa được kiểm tra theo cấu trúc từng nguồn: ngày đối với chuỗi thời gian, đồng tiền–ngày đối với tỷ giá, ISO3–năm đối với chỉ tiêu quốc gia và `time` trong từng CSV tài sản Coin Metrics. Tên CSV thành viên cung cấp mã tài sản, nên khóa của toàn bộ dữ liệu crypto là mã tài sản kết hợp với thời gian.

Kết quả được lưu trong `raw_key_checks.csv`. Notebook dừng nếu có dòng thiếu khóa, khóa trùng hoặc tổng số dòng dữ liệu thô thấp hơn 5.000. Các kiểm tra này xác nhận điều kiện cấu trúc ban đầu; việc kiểm tra giá trị và công thức được thực hiện ở các notebook tiếp theo.

In [4]:
checks = []
key_specs = {
    "macro/vix_daily.csv": ["DATE"], "macro/brent_daily.csv": ["Date"],
    "macro/fx_daily.csv": ["Country", "Date"], "macro/sp500_monthly.csv": ["Date"],
    "macro/us10y_monthly.csv": ["Date"], "macro/gold_monthly.csv": ["Date"],
    "macro/cpi_us_monthly.csv": ["Date"], "country/wb_cpi.csv": ["Country Code", "Year"],
    "country/wb_gdp.csv": ["Country Code", "Year"], "country/wb_population.csv": ["Country Code", "Year"],
    "country/country_codes.csv": ["ISO3166-1-Alpha-3"],
}
for rel, key in key_specs.items():
    path = RAW / rel
    frame = pd.read_csv(path, low_memory=False, keep_default_na=False if path.name == "country_codes.csv" else True)
    null_keys = int(frame[key].replace("", pd.NA).isna().any(axis=1).sum())
    duplicate_keys = int(frame.duplicated(key).sum())
    checks.append({"raw_path": rel, "key": ",".join(key), "rows": len(frame), "null_key_rows": null_keys, "duplicate_keys": duplicate_keys})
for ticker, frame in iter_coinmetrics():
    checks.append({
        "raw_path": f"crypto/{ticker}", "key": "time", "rows": len(frame),
        "null_key_rows": int(frame["time"].isna().sum()), "duplicate_keys": int(frame.duplicated(["time"]).sum()),
    })
key_checks = pd.DataFrame(checks)
write_csv_atomic(key_checks, QUALITY / "raw_key_checks.csv")
if key_checks[["null_key_rows", "duplicate_keys"]].to_numpy().sum() != 0:
    raise AssertionError(key_checks.query("null_key_rows > 0 or duplicate_keys > 0"))
assert inventory.rows.sum() >= 5000, "Dataset không đạt yêu cầu tối thiểu 5.000 dòng."
key_checks.describe(include="all")

,raw_path,key,rows,null_key_rows,duplicate_keys
count,74,74,74.000000,74.0,74.0
unique,74,6,NaN,NaN,NaN
top,macro/vix_daily.csv,time,NaN,NaN,NaN
freq,1,63,NaN,NaN,NaN
mean,NaN,NaN,7162.405405,0.0,0.0
std,NaN,NaN,31469.990949,0.0,0.0
min,NaN,NaN,249.000000,0.0,0.0
25%,NaN,NaN,2125.500000,0.0,0.0
50%,NaN,NaN,3096.500000,0.0,0.0
75%,NaN,NaN,3571.000000,0.0,0.0
